In [4]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
import polars
from skforecast.datasets import fetch_dataset
import sys

# Plots
# ==============================================================================
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)

pd.set_option('display.float_format', '{:.6f}'.format)

Load dataframe with all model predictions

In [8]:
df_pred = pd.read_parquet('/Users/Georgi/Dropbox/Data and AI/Applications/Group Project Datasets/df_pred20241120.parquet', engine='pyarrow')
df_test = pd.read_parquet('/Users/Georgi/Dropbox/Data and AI/Applications/Group Project Datasets/df_test_with_forecasts_seperate_20241122.parquet', engine='pyarrow')
# df_val = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/df_val_with_forecasts_20241120.parquet', engine='pyarrow')

In [9]:
df_test.head()

,,,y_pred
date,store_nbr,item_nbr,
2017-02-20,1,103520,12.948643
2017-02-27,1,103520,18.878524
2017-03-06,1,103520,17.480620
2017-03-13,1,103520,19.921091
2017-03-20,1,103520,14.294333


In [10]:
df_test = df_test.reset_index()

df_test.head()

,date,store_nbr,item_nbr,y_pred
0,2017-02-20,1,103520,12.948643
1,2017-02-27,1,103520,18.878524
2,2017-03-06,1,103520,17.480620
3,2017-03-13,1,103520,19.921091
4,2017-03-20,1,103520,14.294333


In [11]:
df_test.sort_values(by=['store_nbr', 'item_nbr', 'date'])

,date,store_nbr,item_nbr,y_pred
0,2017-02-20,1,103520,12.948643
1,2017-02-27,1,103520,18.878524
2,2017-03-06,1,103520,17.480620
3,2017-03-13,1,103520,19.921091
4,2017-03-20,1,103520,14.294333
...,...,...,...,...
909631,2017-07-17,54,2089339,0.088981
909632,2017-07-24,54,2089339,0.088981
909633,2017-07-31,54,2089339,0.088981
909634,2017-08-07,54,2089339,0.088981


In [12]:
df_pred.sort_values(by=['store_nbr', 'item_nbr', 'date'])

,store_nbr,item_nbr,unit_sales,week_number_cum,date,unique_id,y_naive,y_mean,store_cluster,item_class,perishable,store_type,item_family,y_xgb
173446,1,103520,9.333333,217,2017-02-20,6671,9.500000,10.800000,13,1028,0,D,GROCERY I,12.308541
173447,1,103520,11.333333,218,2017-02-27,6671,19.400000,13.300000,13,1028,0,D,GROCERY I,12.420851
173448,1,103520,14.200000,219,2017-03-06,6671,9.333333,12.744444,13,1028,0,D,GROCERY I,13.872335
173449,1,103520,15.200000,220,2017-03-13,6671,11.333333,13.355555,13,1028,0,D,GROCERY I,12.880795
173450,1,103520,42.266666,221,2017-03-20,6671,14.200000,11.622222,13,1028,0,D,GROCERY I,13.544174
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
791929,54,2089339,36.000000,238,2017-07-17,30458,35.571430,31.857143,3,1006,0,C,GROCERY I,25.557020
791930,54,2089339,47.000000,239,2017-07-24,30458,37.000000,33.523810,3,1006,0,C,GROCERY I,23.175959
791931,54,2089339,30.000000,240,2017-07-31,30458,36.000000,36.190477,3,1006,0,C,GROCERY I,25.630396
791932,54,2089339,31.000000,241,2017-08-07,30458,47.000000,40.000000,3,1006,0,C,GROCERY I,29.772964


In [14]:
df_pred['y_holt'] = df_test['y_pred']

In [15]:
df = df_pred
df.head()

,store_nbr,item_nbr,unit_sales,week_number_cum,date,unique_id,y_naive,y_mean,store_cluster,item_class,perishable,store_type,item_family,y_xgb,y_holt
0,10,1001305,0.000000,217,2017-02-20,0,0.000000,0.000000,15,1016,0,C,GROCERY I,0.275311,12.948643
1,10,1001305,0.000000,218,2017-02-27,0,0.000000,0.000000,15,1016,0,C,GROCERY I,1.186699,18.878524
2,10,1001305,0.000000,219,2017-03-06,0,0.000000,0.000000,15,1016,0,C,GROCERY I,0.000000,17.480620
3,10,1001305,0.000000,220,2017-03-13,0,0.000000,0.000000,15,1016,0,C,GROCERY I,0.000000,19.921091
4,10,1001305,0.000000,221,2017-03-20,0,0.000000,0.000000,15,1016,0,C,GROCERY I,0.000000,14.294333


Below put in values for holt-winters predictions:

In [ ]:
# df['y_holt'] = np.random.uniform(0, 75, size=len(df))
# df.head()

In [ ]:
# df = df[df['week_number_cum'] != 242]

In [ ]:
# def calculate_mape(actual, predicted):
#     actual = np.array(actual)
#     predicted = np.array(predicted)
    
#     # Avoid division by zero by replacing zero actual values with a small number (epsilon)
#     epsilon = 1e-10
#     actual = np.where(actual == 0, epsilon, actual)
    
#     mape = np.mean(np.abs((actual - predicted) / actual)) * 100
#     return mape

# # Example usage
# actual = [0, 10, 20, 30]
# predicted = [5, 10, 25, 35]
# mape = calculate_mape(df['unit_sales'], df['y_mean'])
# print(f"MAPE: {mape:.2f}%")


Define functions for 
- calculating MAPE, 
- calculating bias and accuracy per item, 
- create dataframe with metrics (including costs),
- create dataframe with metrics per group (for example per store_type, or item_family),
- calculate summarized data (sum or mean)

In [16]:
def calculate_mape(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)
    
    # Filter out zero actual values
    mask = actual != 0
    actual = actual[mask]
    predicted = predicted[mask]
    
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return mape

def calculate_bias_acc(df, prediction, actual):
    """
    Calculate bias, accuracy, and adjusted bias for predictions.
    
    Parameters:
        df (pd.DataFrame): The DataFrame containing the prediction and actual data.
        prediction (str): Column name for the predictions.
        actual (str): Column name for the actual values.

    Returns:
        pd.DataFrame: The DataFrame with new columns for bias, accuracy, and adjusted bias.
    """
    # Calculate bias
    df[f'bias_{prediction}'] = df[prediction] - df[actual]
    
    # Calculate accuracy
    df[f'acc_{prediction}'] = np.where(
        (df[actual] == 0) & (df[prediction] != 0),
        np.nan,
        (1 - np.abs(df[f'bias_{prediction}']) / df[actual]) * 100
    )
    
    # Calculate adjusted bias
    df[f'adj_bias_{prediction}'] = (df[f'bias_{prediction}'] * 2 / 3).where(
        df[f'bias_{prediction}'] >= 0, 
        df[f'bias_{prediction}'] * -1 / 3
    )
    
    return df

def calculate_metrics(df):
    # Calculate mean accuracies
    accuracy_xgb = np.mean(df['acc_y_xgb'])
    accuracy_naive = np.mean(df['acc_y_naive'])
    accuracy_mean = np.mean(df['acc_y_mean'])
    accuracy_holt = np.mean(df['acc_y_holt'])
        
    # Calculate bias and positive/negative bias for XGB and Naive models
    bias_xgb = np.mean(df['bias_y_xgb'])
    positive_bias_xgb = np.mean(df['bias_y_xgb'][df['bias_y_xgb'] > 0])
    negative_bias_xgb = np.mean(df['bias_y_xgb'][df['bias_y_xgb'] < 0])
    
    bias_naive = np.mean(df['bias_y_naive'])
    positive_bias_naive = np.mean(df['bias_y_naive'][df['bias_y_naive'] > 0])
    negative_bias_naive = np.mean(df['bias_y_naive'][df['bias_y_naive'] < 0])

    bias_mean = np.mean(df['bias_y_mean'])
    positive_bias_mean = np.mean(df['bias_y_mean'][df['bias_y_mean'] > 0])
    negative_bias_mean = np.mean(df['bias_y_mean'][df['bias_y_mean'] < 0])

    bias_holt = np.mean(df['bias_y_holt'])
    positive_bias_holt = np.mean(df['bias_y_holt'][df['bias_y_holt'] > 0])
    negative_bias_holt = np.mean(df['bias_y_holt'][df['bias_y_holt'] < 0])

    # Calculate sum of adjusted bias
    adj_bias_xgb = np.sum(df['adj_bias_y_xgb'])
    adj_bias_naive = np.sum(df['adj_bias_y_naive'])
    adj_bias_mean = np.sum(df['adj_bias_y_mean'])
    adj_bias_holt = np.sum(df['adj_bias_y_holt'])
    diff_xgb_naive = adj_bias_naive - adj_bias_xgb
    diff_mean_naive = adj_bias_naive - adj_bias_mean
    diff_holt_naive = adj_bias_naive - adj_bias_holt

    # Calculate MAPE
    mape_xgb = calculate_mape(df['unit_sales'], df['y_xgb'])
    mape_naive = calculate_mape(df['unit_sales'], df['y_naive'])
    mape_mean = calculate_mape(df['unit_sales'], df['y_mean'])
    mape_holt = calculate_mape(df['unit_sales'], df['y_holt'])
       
    # Store results in a dictionary
    results = {
        'Metric': [
            'Mean Accuracy XGB', 'Mean Accuracy Naive',
            'Mean Accuracy Mean', 'Mean Accuracy Holt',
            'MAPE XGB', 'MAPE Naive', 'MAPE Mean', 'MAPE Holt',
            'Bias XGB', 'Positive Bias XGB', 'Negative Bias XGB', 
            'Bias Naive', 'Positive Bias Naive', 'Negative Bias Naive',
            'Bias Mean', 'Positive Bias Mean', 'Negative Bias Mean',
            'Bias Holt', 'Positive Bias Holt', 'Negative Bias Holt',
            'Money XGB', 'Money Naive', 'Money Mean', 'Money Holt',
            'Difference Naive - XGB', 'Difference Naive - Mean', 'Difference Naive - Holt'
        ],
        'Value': [
            accuracy_xgb, accuracy_naive, 
            accuracy_mean, accuracy_holt,
            mape_xgb, mape_naive, mape_mean, mape_holt,
            bias_xgb, positive_bias_xgb, negative_bias_xgb,
            bias_naive, positive_bias_naive, negative_bias_naive,
            bias_mean, positive_bias_mean, negative_bias_mean,
            bias_holt, positive_bias_holt, negative_bias_holt,
            adj_bias_xgb, adj_bias_naive, adj_bias_mean, adj_bias_holt,
            diff_xgb_naive, diff_mean_naive, diff_holt_naive
        ]
    }
    
    # Convert results dictionary to a DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df

def calculate_metrics_grouped(df, column):
    store_stats = []
    
    # Iterate over each group's data
    for store, group in df.groupby(column):
        # # Skip calculation if the group value is 0 or missing (NaN)
        # if store == 0 or pd.isna(store):
        #     continue
        
        # Calculate statistics for the current group using the existing function
        df_metrics = calculate_metrics(group)
        
        # Add the column value (e.g., store number) to each metric's result for reference
        df_metrics[column] = store
        
        # Append each group's result to the list
        store_stats.append(df_metrics)
    
    # Concatenate all group statistics into a single DataFrame
    store_means = pd.concat(store_stats).reset_index(drop=True)

    # Pivot the DataFrame to make column values into columns and Metric rows
    store_metrics_pivot = store_means.pivot(index='Metric', columns=column, values='Value')

    # Drop columns with all NA values
    store_metrics_pivot = store_metrics_pivot.dropna(axis=1, how='all')

    return store_metrics_pivot

def summarize_store_data(df, group_column, summary_columns, operation='sum'):
    if operation == 'sum':
        grouped_df = df.groupby(group_column)[summary_columns].sum().reset_index()
    elif operation == 'mean':
        grouped_df = df.groupby(group_column)[summary_columns].mean().reset_index()
    else:
        raise ValueError("Invalid operation. Choose 'sum' or 'mean'.")
    
    return grouped_df

Run function 'calculate_bias_acc' for every column containing the predictions of a model, listed in the list_pred

In [17]:
list_pred = ['y_xgb', 'y_mean', 'y_naive', 'y_holt']

# Apply the function for every prediction column in list_pred
for pred in list_pred:
    df = calculate_bias_acc(df, pred, 'unit_sales')

# Display the updated DataFrame
df.head()

,store_nbr,item_nbr,unit_sales,week_number_cum,date,unique_id,y_naive,y_mean,store_cluster,item_class,...,adj_bias_y_xgb,bias_y_mean,acc_y_mean,adj_bias_y_mean,bias_y_naive,acc_y_naive,adj_bias_y_naive,bias_y_holt,acc_y_holt,adj_bias_y_holt
0,10,1001305,0.000000,217,2017-02-20,0,0.000000,0.000000,15,1016,...,0.183541,0.000000,NaN,0.000000,0.000000,NaN,0.000000,12.948643,NaN,8.632429
1,10,1001305,0.000000,218,2017-02-27,0,0.000000,0.000000,15,1016,...,0.791133,0.000000,NaN,0.000000,0.000000,NaN,0.000000,18.878524,NaN,12.585683
2,10,1001305,0.000000,219,2017-03-06,0,0.000000,0.000000,15,1016,...,0.000000,0.000000,NaN,0.000000,0.000000,NaN,0.000000,17.480620,NaN,11.653747
3,10,1001305,0.000000,220,2017-03-13,0,0.000000,0.000000,15,1016,...,0.000000,0.000000,NaN,0.000000,0.000000,NaN,0.000000,19.921091,NaN,13.280727
4,10,1001305,0.000000,221,2017-03-20,0,0.000000,0.000000,15,1016,...,0.000000,0.000000,NaN,0.000000,0.000000,NaN,0.000000,14.294333,NaN,9.529555


Overview of metrics:

In [18]:
df_metrics = calculate_metrics(df)
df_metrics

,Metric,Value
0,Mean Accuracy XGB,45.506119
1,Mean Accuracy Naive,36.467354
2,Mean Accuracy Mean,38.832172
3,Mean Accuracy Holt,-151.920550
4,MAPE XGB,54.493892
5,MAPE Naive,63.532662
6,MAPE Mean,61.167828
7,MAPE Holt,251.920550
8,Bias XGB,1.764386
9,Positive Bias XGB,15.494104


Show metrics grouped by column, for example store_type, item_family. All possibilities are shown in list_grouping

In [19]:
list_grouping = ['store_type', 'store_cluster', 'store_nbr', 'perishable', 'item_family', 'item_class', 'week_number_cum']

# Initialize an empty dictionary
dict_df_metrics = {}

for i in list_grouping:
    # Calculate metrics for the current grouping
    df_metrics_grouped = calculate_metrics_grouped(df, i)
    dict_df_metrics[i] = df_metrics_grouped  # Add the resulting DataFrame to the dictionary with `i` as the key
    print(df_metrics_grouped)

store_type                            A               B               C  \
Metric                                                                    
Bias Holt                    -21.046883       -7.970001       14.350115   
Bias Mean                      2.551649        1.017984        0.522350   
Bias Naive                     2.451751        1.146251        0.598703   
Bias XGB                       4.643422        1.520324       -0.194850   
Difference Naive - Holt -3540438.898867 -1779326.454626 -4571185.079456   
Difference Naive - Mean   131465.454238    81463.229099    84053.568251   
Difference Naive - XGB     55979.250000   155853.625000   292085.000000   
MAPE Holt                    196.165392      184.726445      292.472455   
MAPE Mean                     59.802602       60.603509       65.198753   
MAPE Naive                    61.150807       64.976215       67.794752   
MAPE XGB                      55.267364       52.736354       55.859476   
Mean Accuracy Holt       

Show sum of sales, predictions, accuracy and bias - per group, listed in list_groups

In [ ]:
for i in list_grouping:
    df_sum = summarize_store_data(df, group_column=i, 
                              summary_columns=['unit_sales', 'y_naive', 'y_xgb', 'y_mean', 'y_holt',  
                                               'acc_y_xgb', 'acc_y_naive', 'acc_y_mean', 'acc_y_holt', 
                                               'adj_bias_y_xgb', 'adj_bias_y_naive', 'adj_bias_y_mean', 'adj_bias_y_holt'],
                              operation='sum')
    print(df_sum)

Show mean of sales, predictions, accuracy and bias - per group, listed in list_groups


In [ ]:
for i in list_grouping:
    df_mean = summarize_store_data(df, group_column=i, 
                              summary_columns=['unit_sales', 'y_naive', 'y_xgb', 'y_mean', 'y_holt',  
                                               'acc_y_xgb', 'acc_y_naive', 'acc_y_mean', 'acc_y_holt', 
                                               'adj_bias_y_xgb', 'adj_bias_y_naive', 'adj_bias_y_mean', 'adj_bias_y_holt'],
                              operation='mean')
    print(df_mean)

Define functions to create visualisations:
- Histograms to show different in cost compared to naive, for every model. This histogram is sorted by biggest difference
- Linegraph to show sales for different models (possibility to group on a column)

In [ ]:
def sorted_histogram(dict_df, group_names):
    """
    Plots sorted histograms for the first row (sorted) and unsorted histograms for others.
    The bars for each row are plotted next to each other.

    Parameters:
        dict_df (dict): A dictionary where keys are group names and values are DataFrames.
        group_names (list): A list of row names to plot.

    Returns:
        None
    """
    # Iterate over each DataFrame in the dictionary
    for key, df in dict_df.items():
        print(f"Processing group: {key}")
        
        # Create a new figure for each group
        plt.figure(figsize=(12, 8))
        
        # Get the number of elements in the first row to define x positions
        # (Assumes all rows have the same length, but you could adjust for mismatched lengths)
        num_elements = len(df.loc[group_names[0]])
        
        # Initialize the width of each bar and set the starting position for the bars
        bar_width = 0.2
        index = np.arange(num_elements)  # positions for the x-axis
        
        # Loop through each group name in the provided list
        for i, group_name in enumerate(group_names):
            # Ensure the row exists in the DataFrame
            if group_name not in df.index:
                raise ValueError(f"Row '{group_name}' not found in the DataFrame index of group '{key}'.")
            
            # Extract the specified row and convert to a Series
            difference_row = df.loc[group_name]
            
            # Sort the first group (group_names[0]) only
            if i == 0:
                difference_row = difference_row.sort_values(ascending=False)
            
            # Plot the histogram for each group_name, shifting bars by `i * bar_width`
            # This ensures the bars for different rows are placed next to each other
            plt.bar(index + i * bar_width, difference_row, width=bar_width, label=group_name)
        
        # Add title, labels, and grid
        plt.title(f'Saved money compared to naive, per: {key}', fontsize=14)
        plt.xlabel({key}, fontsize=12)
        plt.ylabel('Saved money', fontsize=12)
        
        # Adjust x-ticks to correspond to the center of the grouped bars
        plt.xticks(index + bar_width * (len(group_names) - 1) / 2, difference_row.index.astype(str), rotation=90)
        
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.legend(title="Rows", fontsize=10)
        
        # Adjust layout and show the plot
        plt.tight_layout()
        plt.show()

In [ ]:
def plot_sales_comparison(df, filter_column, filter_value):
    """
    Plots a comparison of predicted sales (y_xgb), actual sales (unit_sales), and naive predictions (y_naive)
    over weeks for a specified store (or any other column-based filter).

    Parameters:
    df_pred (pd.DataFrame): DataFrame containing the data.
    filter_column (str): The column name to filter by (e.g., 'store_nbr').
    filter_value (int or str): The value to filter the specified column by (e.g., a store number).
    """
    # Step 1: Filter data for the specified column and value
    grouped_df = df[df[filter_column] == filter_value]

    # Step 2: Group by 'week_number_cum' and compute the sum for each of the columns
    grouped_df = grouped_df.groupby('week_number_cum')[['y_xgb', 'unit_sales', 'y_naive', 'y_mean', 'y_holt']].sum().reset_index()

    # Step 3: Plot the results
    plt.figure(figsize=(12, 6))
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_xgb'], marker='o', label='XGB prediction (y_xgb)', color='blue')
    plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='black')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_naive'], marker='o', label='Naive prediction (y_naive)', color='red')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_mean'], marker='o', label='Average mean prediction (y_mean)', color='orange')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_holt'], marker='o', label='Holt Winters prediction (y_holt)', color='green')

    # Add titles and labels
    plt.title(f'Sum of Predicted sales vs Actual sales Over Weeks for {filter_column} = {filter_value}')
    plt.xlabel('Week Number Cumulative')
    plt.ylabel('Sales')
    plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
    plt.legend()
    plt.grid()

    # Show the plot
    plt.tight_layout()
    plt.show()


Line graph for total sales:

In [ ]:
    grouped_df = df
    # Step 2: Group by 'week_number_cum' and compute the sum for each of the columns
    grouped_df = grouped_df.groupby('week_number_cum')[['y_xgb', 'unit_sales', 'y_naive', 'y_mean', 'y_holt']].sum().reset_index()

    # Step 3: Plot the results
    plt.figure(figsize=(12, 12))
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_xgb'], marker='o', label='XGB prediction (y_xgb)', color='blue')
    plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='black')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_naive'], marker='o', label='Naive prediction (y_naive)', color='red')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_mean'], marker='o', label='Average mean prediction (y_mean)', color='orange')
    plt.plot(grouped_df['week_number_cum'], grouped_df['y_holt'], marker='o', label='Holt Winters prediction (y_holt)', color='green')

    # Add titles and labels
    plt.title(f'Sum of Predicted sales vs Actual sales Over Weeks')
    plt.xlabel('Week Number Cumulative')
    plt.ylabel('Sales')
    plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
    plt.legend()
    plt.grid()

    # Show the plot
    plt.tight_layout()
    plt.show()

Line graph for total sales, grouped by a column, for example item_family, store_type etc...

In [ ]:
# Choose from list_grouping a column, get unique values, and create line graphs with total actual sales, predicted sales and naive prediction

column = 'item_family'
# Get unique values in the current column
unique_values = df[column].unique()
    
# Loop through each unique value in the column
for value in unique_values:
    # Call the plot_sales_comparison function
    plot_sales_comparison(df, filter_column=column, filter_value=value)


Plot the difference between naive and a model in histograms per group

In [ ]:
group_names = ['Difference Naive - XGB', 'Difference Naive - Mean', 'Difference Naive - Holt']
sorted_histogram(dict_df_metrics, group_names)

Line diagram of the cumulative saved costs compared to naive, per model

In [ ]:
df_week_number_cum = calculate_metrics_grouped(df, 'week_number_cum')

# Assuming df_plot is the DataFrame you are working with
df_plot = df_week_number_cum

# Extract the row for "Difference Naive - XGB" and transpose it to work with it as a Series
difference_row_xgb = df_plot.loc['Difference Naive - XGB'].transpose()

# Extract the row for "Difference Naive - Mean" and transpose it to work with it as a Series
difference_row_mean = df_plot.loc['Difference Naive - Mean'].transpose()

# Extract the row for "Difference Naive - Holt" and transpose it to work with it as a Series
difference_row_holt = df_plot.loc['Difference Naive - Holt'].transpose()

# Ensure the index represents week numbers and the values are numeric
week_numbers = df_plot.columns.astype(int)  # Assuming column names are week numbers

# Convert the values for both rows to numeric, coercing any errors to NaN
difference_values_xgb = pd.to_numeric(difference_row_xgb, errors='coerce')
difference_values_mean = pd.to_numeric(difference_row_mean, errors='coerce')
difference_values_holt = pd.to_numeric(difference_row_holt, errors='coerce')

# Compute the cumulative sum of the difference values for both rows
cumulative_difference_xgb = difference_values_xgb.cumsum()
cumulative_difference_mean = difference_values_mean.cumsum()
cumulative_difference_holt = difference_values_holt.cumsum()

# Plot the cumulative difference for both "Difference Naive - XGB" and "Difference Naive - Mean"
plt.figure(figsize=(12, 6))

# Plot the cumulative difference for "Difference Naive - XGB"
plt.plot(week_numbers, cumulative_difference_xgb, marker='o', label='Money saved by XGB model', color='blue')

# Plot the cumulative difference for "Difference Naive - Mean"
plt.plot(week_numbers, cumulative_difference_mean, marker='s', label='Money saved by moving average model', color='orange')

# Plot the cumulative difference for "Difference Naive - Mean"
plt.plot(week_numbers, cumulative_difference_holt, marker='s', label='Money saved by Holt Winters model', color='green')

# Add labels and title
plt.title('Cumulative Money Saved by model', fontsize=14)
plt.xlabel('Week Number', fontsize=12)
plt.ylabel('Total Money Saved', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Show the plot
plt.tight_layout()
plt.show()
